# ⚡ Dask: Faster-than-Pandas (CPU Benchmark)

### **Template Review**
This template demonstrates how to handle **larger-than-memory** datasets using **Dask** on a standard CPU environment. Optimized for **Saturn Cloud Jupyter Notebooks**, it provides a direct performance comparison between standard Pandas (single-core) and Dask (multi-core parallel processing).

**Core Workflow:** We will generate a massive **2GB+ Synthetic Dataset** locally with complex data types (Dates, Strings). This ensures the workload is heavy enough to make Pandas struggle, highlighting Dask's ability to process data in chunks without crashing memory.

### **Tech Stack**
* **Dask**: Parallel computing library for scaling Python.
* **Pandas**: Standard data analysis library (the baseline).
* **Infrastructure**: [Saturn Cloud](https://saturncloud.io/) CPU Jupyter Instance.

In [ ]:
# Install Dask with complete dependencies (Dashboard + Distributed)
# Quotes are used to prevent ZSH/Shell errors with brackets
!pip install "dask[complete]" pandas numpy -q

### **Step 1: Generate Massive Synthetic Data (2GB)**
We generate **20 Million rows** of complex data. We include **Dates** and **String Categories** because these data types are computationally expensive and slow down Pandas significantly.

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

FILENAME = "heavy_data.csv"
NUM_ROWS = 20_000_000  # 20 Million Rows
CHUNK_SIZE = 1_000_000 # Process in chunks to save memory during creation

def generate_data():
    print(f"🔨 Generating {NUM_ROWS} rows (Approx 2GB)... Please wait.")
    
    # Categories for string complexity (Strings are slower than Ints)
    categories = ['Electronics', 'Furniture', 'Clothing', 'Food', 'Auto']
    start_date = datetime(2020, 1, 1)
    
    # Remove old file if exists
    if os.path.exists(FILENAME):
        os.remove(FILENAME)
        
    for i in range(0, NUM_ROWS, CHUNK_SIZE):
        # Create complex data: Dates + Strings + Floats
        df = pd.DataFrame({
            'transaction_id': np.arange(i, i + CHUNK_SIZE),
            'date': [start_date + timedelta(days=x % 365) for x in range(CHUNK_SIZE)],
            'category': np.random.choice(categories, CHUNK_SIZE),
            'amount': np.random.uniform(10.0, 500.0, CHUNK_SIZE),
            'discount': np.random.uniform(0.0, 0.3, CHUNK_SIZE)
        })
        
        # Append to CSV
        mode = 'a' if i > 0 else 'w'
        header = (i == 0)
        df.to_csv(FILENAME, index=False, mode=mode, header=header)
        
        if (i // CHUNK_SIZE) % 5 == 0:
            print(f"... Written {(i + CHUNK_SIZE) / 1_000_000:.0f} Million rows")
            
    print(f"✅ Done! File size: {os.path.getsize(FILENAME) / (1024**3):.2f} GB")

# Check if file exists and is big enough (>1.5GB)
if not os.path.exists(FILENAME) or os.path.getsize(FILENAME) < 1.5 * 1024**3:
    generate_data()
else:
    print("✅ Large file already exists. Skipping generation.")

### **Step 2: Initialize Dask Client**
Dask uses a "Client" to manage parallel workers. Running this cell will provide a **Dashboard Link** where you can watch your CPU cores light up in real-time.

In [ ]:
from dask.distributed import Client
import dask.dataframe as dd
import time

# Start a local Dask cluster using all available CPU cores
client = Client()
print(f"🚀 Dask Dashboard available at: {client.dashboard_link}")
client

### **Step 3: The Pandas Benchmark (The Struggle)**
We attempt to read the 2GB file and parse dates into memory. 

> **Note:** We use `parse_dates` specifically because it is an expensive operation that forces Pandas to work hard. This step usually takes **60-90 seconds** or triggers a MemoryError on smaller machines.

In [ ]:
start_time = time.time()
print("🐢 Pandas: Reading & Processing...")

try:
    # We force date parsing to ensure CPU load is high
    df_pd = pd.read_csv(FILENAME, parse_dates=['date'])
    
    # Complex GroupBy Operation
    print("🐢 Pandas: Grouping & Aggregating...")
    res_pd = df_pd.groupby('category')['amount'].mean()
    
    pd_duration = time.time() - start_time
    print(f"⏱️ Pandas Time: {pd_duration:.2f} seconds")
except MemoryError:
    print("❌ Pandas Crashed (MemoryError)! This proves the data is too big for RAM.")
    pd_duration = float('inf')

### **Step 4: The Dask Benchmark (The Solution)**
Dask processes this **lazily**. It scans the file structure instantly and then processes chunks in parallel across all your CPU cores. It never loads the entire file at once.

In [ ]:
start_time = time.time()
print("🐇 Dask: Lazy Read & Parallel Compute...")

# Dask handles date parsing efficiently across cores
ddf = dd.read_csv(FILENAME, parse_dates=['date'])

# .compute() triggers the actual parallel execution
res_dask = ddf.groupby('category')['amount'].mean().compute()

dask_duration = time.time() - start_time
print(f"⏱️ Dask Time: {dask_duration:.2f} seconds")

# Calculate Speedup Factor
if pd_duration != float('inf'):
    print(f"\n🚀 Speedup: {pd_duration / dask_duration:.2f}x Faster")
else:
    print("\n🏆 Dask Wins (Pandas Crashed)")

---
## 🏁 Conclusion & Next Steps
We have successfully demonstrated that **Dask** can handle heavy datasets that slow down standard Pandas workflows. By parallelizing the `read_csv` and `groupby` operations, Dask maximizes the utility of your **Saturn Cloud CPU** instance.

### **Resources & Backlinks**
* **Cloud Infrastructure**: [Deploy on Saturn Cloud](https://saturncloud.io/)
* **Dask Documentation**: [Best Practices](https://docs.dask.org/en/stable/best-practices.html)